# Score claims and explain one flag

Origami Health · synthetic training only. Website: lab3. Planned block: 15 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Use a prepared model to prioritize review and explain why a high score is not proof of fraud.

**Before you begin:** Your Day 1 Gold data is validated, or the facilitator has explicitly provided and labelled a validated recovery snapshot. The model is already prepared; training is not part of this 15-minute lab.

**Key terms:** A feature is an input to a model. Scoring applies learned weights to new rows. A threshold converts a score into a review priority. A false positive is a legitimate case flagged for review. Registration stores a model version in a registry; a JSON file alone is not a registered model.

**Timebox:** This notebook belongs to the 15-minute website block. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 2 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell and confirm your assigned slot.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Score with the prepared model; no training in class

**Purpose:** Score with the prepared model; no training in class

**Inputs:** Persisted Gold claim detail and prepared JSON model.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** 720 scored claims; C0601 illustrates a legitimate high-value flag. Registry parity remains a separate admin gate.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 2:** Run the scoring cell. Inspect one high-ranked claim and its reason_codes. Then inspect C0601: it is deliberately legitimate but high-value. Explain what the investigator should do next without declaring fraud or rejecting coverage.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
model=json.loads((Path(asset_root)/'model'/'prepared_model.json').read_text())
claims=bounded_rows(read_delta(spark,cfg,'gold_claim_detail'))
scores=score_rows(claims,model)
from pyspark.sql.types import StructType,StructField,StringType,DoubleType
schema=StructType([StructField(k,DoubleType() if k=='review_score' else StringType(),False) for k in scores[0]])
frame=spark.createDataFrame(scores,schema)
saved=persist(spark,cfg,'gold_claim_risk_current',frame,['claim_id'])
verify_fixture(spark,cfg,'gold_claim_risk_current',saved)
saved.orderBy(F.desc('review_score')).show(5,truncate=False)
saved.where(F.col('claim_id')=='C0601').show(truncate=False)
print('C0601 is a deliberately legitimate, high-value synthetic example. A review flag is not proof of fraud.')
save_evidence(cfg,'scoring',{'rows':len(scores),'model_version':model['version'],'model_digest':model['artifact_digest'],'registered_model_parity':'NOT VERIFIED; administrator gate'})

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: Exactly 720 unique claim scores are persisted and match the prepared model’s expected fingerprint. C0601 has REVIEW priority. The output explicitly reports that registered-model parity is not verified by this cell.
- [ ] I saved evidence: Record model version, one selected claim, reason codes and your human-review interpretation. The notebook saves scoring.json. Do not describe local metrics as real customer performance.

**Discuss:** Which inputs drive the model, and what information must not be used as a scoring feature?

**Self-check explanation:** Inputs are log submitted amount, missing-attachment signal and a pre-existing duplicate signal. Future review outcomes, claim denial and final investigation labels are not scoring inputs. Reason codes are explanations of signals, not causal proof.

**Next:** day2/06_benefits_knowledge_base.md. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
